# NB02 — merge shard caches + data-quality gates (CPU)

Inputs: the outputs of the three NB01 shard kernels. Writes the single feature cache used by every training notebook and `qc_report.json` (window counts, durations, ICA removals, θ/α slowing check, PSWE burden per group).

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
import glob
shards = sorted(glob.glob('/kaggle/input/**/cache_s*', recursive=True))
logs = sorted(glob.glob('/kaggle/input/**/preproc/preprocess_log.jsonl', recursive=True))
part = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
print(shards, logs, part, sep='\n')
assert len(shards) == 3 and len(logs) == 3, 'missing NB01 shard outputs'
shutil.copy(part, f'{WORK}/participants.tsv')

In [ ]:
sh(f'{PY} merge-caches --inputs {" ".join(shards)} --out {WORK}/cache', log=f'{WORK}/merge.log')

In [ ]:
sh(f'{PY} qc --cache {WORK}/cache --logs {" ".join(logs)} --out {WORK}/qc_report.json --no-fail')
print(json.load(open(f'{WORK}/qc_report.json'))['gate_failures'])

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)